System Dependencies
To get started with Unstructured.io, we need a few system-wide dependencies:

Poppler (poppler-utils)
Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

Tesseract (tesseract-ocr)
Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

libmagic
File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [1]:
# For Windows:
# You need to install python-magic-bin instead of libmagic
%pip install -Uq python-magic-bin

# IMPORTANT MANUAL STEPS FOR WINDOWS:
# 1. Tesseract OCR: Download and install from https://github.com/UB-Mannheim/tesseract/wiki
#    - Add 'C:\Program Files\Tesseract-OCR' to your System PATH
# 2. Poppler: Download from https://github.com/oschwartz10612/poppler-windows/releases/
#    - Extract it and add the 'bin' folder to your System PATH


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%pip install -Uq "unstructured[pdf]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community
%pip install -Uq langchain-cohere langchain-groq  # <--- Our new fast models!
%pip install -Uq python_dotenv
%pip install -Uq langchain-openai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

# Our Fast Cloud Models
from langchain_cohere import CohereEmbeddings
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI


load_dotenv()

# Initialize the models (Replacing OpenAI)
# 1. Embeddings (Search Math)
embedding_model = CohereEmbeddings(model="embed-english-v3.0")

# 2. Generative AI (The Answer Engine)
llm = ChatGroq(model="llama-3.3-70b-versatile")


e:\Projects\RAG_Prac\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
e:\Projects\RAG_Prac\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "./pdfs/energy.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: ./pdfs/energy.pdf


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 6651.24it/s]


✅ Extracted 1300 elements


In [5]:
elements[3].to_dict()

{'type': 'NarrativeText',
 'element_id': 'aee336687a1c74cd904b1fde55fd3aec',
 'text': 'One of the three objectives of the UN Secretary General under the Sustainable Energy for All (SE4ALL) initiative is to double the share of renewable energy in the global energy mix by 2030, with an emphasis on promoting sustainable forms of renewable energy.',
 'metadata': {'detection_class_prob': 0.9390647411346436,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(349.5378723144531),
     np.float64(471.3944396972656)),
    (np.float64(349.5378723144531), np.float64(698.3491821289062)),
    (np.float64(2640.7966666666707), np.float64(698.3491821289062)),
    (np.float64(2640.7966666666707), np.float64(471.3944396972656))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-03-27T09:51:26',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 2,
  'file_directory': './pdfs',
  'filename': 'energy.pdf',
  'pare

In [6]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

Found 37 images


{'type': 'Image',
 'element_id': '43596d734da2fe548fe75478d2abf014',
 'text': 'chapter 4  renewable    energy  ',
 'metadata': {'detection_class_prob': 0.5903002023696899,
  'coordinates': {'points': ((np.float64(45.00655746459961),
     np.float64(65.09202575683594)),
    (np.float64(45.00655746459961), np.float64(2844.0654296875)),
    (np.float64(2883.33203125), np.float64(2844.0654296875)),
    (np.float64(2883.33203125), np.float64(65.09202575683594))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-03-27T09:51:26',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './pdfs',
  'filename': 'energy.pdf'}}

In [7]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[1].to_dict()

Found 18 tables


{'type': 'Table',
 'element_id': '2b7bfa061c1c894ea311e670f101423e',
 'text': 'Primary energy supply Final energy consumption • Heat and electricity in form ready for consumption. Advantages • Widely used. • Closer to useful energy output valued by • Based on physical measurement of fuels. end-users • Better balance for directly produced RE. • Different conventions for assumptions on Disadvantages efficiencies means that contribution of RE depends on calculation procedure. • Losses need to be allocated. • Underrepresents directly produced RE.',
 'metadata': {'detection_class_prob': 0.9124767780303955,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(352.6599426269531),
     np.float64(982.4564819335938)),
    (np.float64(352.6599426269531), np.float64(1722.1881103515625)),
    (np.float64(2643.316650390625), np.float64(1722.1881103515625)),
    (np.float64(2643.316650390625), np.float64(982.4564819335938))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layo

In [8]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=2800, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2000, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 106 chunks


In [9]:
# View all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

# View a single chunk
chunks[0].to_dict()

# View original elements
chunks[0].metadata.orig_elements

In [12]:
import os


def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data



def create_ai_enhanced_summary(text: str, tables: list, images: list) -> str:
    """Create AI-enhanced summary using OpenAI GPT-4o-mini for vision"""
    
    try:
        # 1. Initialize the OpenAI Vision model
        vision_llm = ChatOpenAI(model="gpt-4o-mini")
        
        # 2. Build the system/user content array
        content = [
            {
                "type": "text",
                "text": f"""You are creating a searchable description for document content retrieval.
                TEXT CONTENT:
                {text}
                
                TABLES FOUND:
                {'\n'.join(tables) if tables else 'None'}
                """
            }
        ]
        
        # 3. Add images as base64 URLs
        for i, img_b64 in enumerate(images):
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
            })
            
        # 4. Invoke the model
        response = vision_llm.invoke([HumanMessage(content=content)])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ OpenAI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        return summary



def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/106
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: This is a descriptive overview for document retrieval of the provided content:

**Title:** Chapter 4: Renewable Energy

**Summary:**
This document serves as the introductory page for Chapter 4 of a re...
   Processing chunk 2/106
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/106
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/106
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 5/106
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 6/106
     Types found: ['text']
     Tables: 0